# Gestión de Riesgo

> **Contexto de dominio:** [`docs/fuentes/gestion_riesgo.md`](../../docs/fuentes/gestion_riesgo.md)  
> **Bloque:** A | **Línea:** `gestion_riesgo`  
> **Variable principal:** `precipitacion` (mm)  
> **Modelos sugeridos:** SARIMA, SARIMAX, XGBOOST, RANDOM_FOREST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/gestion_riesgo.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "gestion_riesgo"
VARIABLE = "precipitacion"
UNIDAD = "mm"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `gestion_riesgo` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "gestion_riesgo.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/gestion_riesgo.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/gestion_riesgo.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "precipitacion": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: gestion_riesgo -->

## 1b. Componentes AVR — Amenaza, Vulnerabilidad y Riesgo

La **Ley 1523/2012** define el riesgo como la combinacion de amenaza y vulnerabilidad:

```
Riesgo = Amenaza x Vulnerabilidad x Exposicion
Escala: Muy Baja (1) | Baja (2) | Media (3) | Alta (4) | Muy Alta (5)
```

| Fenomeno | Indicador clave | Detonante | Fuente |
|---|---|---|---|
| Inundacion | Profundidad agua (m), caudal | Lluvia extrema | IDEAM/HEC-RAS |
| Movimiento masa | Pendiente, litologia | Lluvia/sismo | SGC |
| Avenida torrencial | IVET (0-1) | Lluvia intensa | SGC/IDEAM |
| Sequia | SPI < -1.5 | Deficit lluvia | IDEAM |

**IVET (Indice de Vulnerabilidad a Eventos Torrenciales):** combina pendiente, cobertura, morfometria de cuenca e indice de Melton. IVET > 0.6 = alta susceptibilidad.

**SPI (Standardized Precipitation Index):** SPI < -1.5 = sequia severa; SPI > 1.5 = lluvia extrema.

In [ ]:
# AVR sintetico por municipio + SPI de precipitacion
np.random.seed(123)
n = len(df)

# -- SPI (Indice de Precipitacion Estandarizado) -------------------------
precip = df['precipitacion'].values
mu_p, std_p = precip.mean(), precip.std()
spi = (precip - mu_p) / std_p  # SPI-1 mensual
df['spi'] = spi.round(3)

# -- IVET por subcuenca (valor anual, escala 0-1) -------------------------
n_subcuencas = 8
subcuencas = [f'SC-{i+1:02d}' for i in range(n_subcuencas)]
ivet = np.random.beta(2, 3, n_subcuencas)  # distribucion tipica IVET

# -- Indice de Amenaza (IA) + Vulnerabilidad (IV) + Riesgo (IR) ----------
# Escala 1-5, simulando evaluacion AVR por municipio
n_municipios = 15
municipios = [f'Mpio-{i+1:02d}' for i in range(n_municipios)]
ia = np.random.randint(1, 6, n_municipios)
iv = np.random.randint(1, 6, n_municipios)
ir = np.clip((ia * iv / 5).astype(int), 1, 5)

df_avr = pd.DataFrame({'municipio': municipios, 'amenaza': ia,
                        'vulnerabilidad': iv, 'riesgo': ir})
df_avr['categoria_riesgo'] = pd.cut(df_avr['riesgo'],
    bins=[0,1,2,3,4,5], labels=['Muy Baja','Baja','Media','Alta','Muy Alta'])

print(f'SPI | min={spi.min():.2f} max={spi.max():.2f}')
print(f'Meses SPI < -1.5 (sequia severa): {(spi < -1.5).sum()}')
print(f'Meses SPI > 1.5 (lluvia extrema): {(spi > 1.5).sum()}')
print(f'\nIVET subcuencas | max={ivet.max():.3f} (alta susceptibilidad torrent si >0.6)')
print(f'Subcuencas IVET > 0.6: {(ivet > 0.6).sum()}/{n_subcuencas}')
df_avr

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `gestion_riesgo` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_gestion_riesgo.html",
       title="EDA — Gestión de Riesgo", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "precipitacion", title="Gestión de Riesgo — precipitacion (mm)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "precipitacion", period="month")
plt.show()

## 3c. Dashboard AVR — Visualizacion de riesgo compuesto

Los cuatro paneles responden preguntas clave del ciclo GRD (Ley 1523/2012):

- **SPI:** deteccion de sequias extremas (detonante hidrologico)
- **IVET:** susceptibilidad a avenidas torrenciales por subcuenca
- **Matriz AVR:** distribucion amenaza vs. vulnerabilidad por municipio
- **Riesgo compuesto:** clasificacion final para POT y POMCA

> **InSAR / DInSAR** (Interferometric SAR): complementa el AVR midiendo deformacion del terreno en milimetros, fundamental para monitorear movimientos en masa activos y zonas de subsidencia. Fuentes: ESA Sentinel-1 (C-SAR) o ALOS-2 PALSAR-2.

In [ ]:
RISK_COLORS = {'Muy Baja':'#27ae60','Baja':'#82e0aa','Media':'#f1c40f',
               'Alta':'#e67e22','Muy Alta':'#e74c3c'}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel A: SPI mensual
ax = axes[0, 0]
colors_spi = ['#e74c3c' if s < -1.5 else '#e67e22' if s < 0
              else '#27ae60' if s > 1.5 else '#82e0aa' for s in df['spi']]
ax.bar(df['fecha'], df['spi'], color=colors_spi, width=20)
ax.axhline(-1.5, color='red', ls='--', lw=1.5, label='Sequia severa SPI<-1.5')
ax.axhline(1.5, color='blue', ls='--', lw=1.5, label='Lluvia extrema SPI>1.5')
ax.set_title('SPI — Indice de Precipitacion Estandarizado', fontweight='bold')
ax.set_ylabel('SPI'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Panel B: IVET por subcuenca
ax = axes[0, 1]
colors_ivet = ['#e74c3c' if v > 0.6 else '#e67e22' if v > 0.4 else '#27ae60'
               for v in ivet]
ax.barh(subcuencas, ivet, color=colors_ivet, alpha=0.85)
ax.axvline(0.6, color='red', ls='--', lw=1.5, label='Alta susceptibilidad IVET>0.6')
ax.set_title('IVET — Susceptibilidad a Avenidas Torrenciales', fontweight='bold')
ax.set_xlabel('IVET (0-1)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

# Panel C: Scatter Amenaza vs Vulnerabilidad
ax = axes[1, 0]
scatter_colors = df_avr['categoria_riesgo'].map(RISK_COLORS)
sc = ax.scatter(df_avr['amenaza'], df_avr['vulnerabilidad'],
                c=scatter_colors, s=80, alpha=0.8, edgecolors='gray', lw=0.5)
ax.set_xlabel('Amenaza (1-5)'); ax.set_ylabel('Vulnerabilidad (1-5)')
ax.set_title('Matriz Amenaza vs. Vulnerabilidad por Municipio', fontweight='bold')
for _, r in df_avr.iterrows():
    ax.annotate(r['municipio'], (r['amenaza'], r['vulnerabilidad']),
                fontsize=6, ha='center', va='bottom')
ax.grid(alpha=0.3)

# Panel D: Distribucion riesgo compuesto
ax = axes[1, 1]
riesgo_counts = df_avr['categoria_riesgo'].value_counts().reindex(
    ['Muy Baja','Baja','Media','Alta','Muy Alta'], fill_value=0)
ax.bar(riesgo_counts.index, riesgo_counts.values,
       color=[RISK_COLORS[k] for k in riesgo_counts.index], alpha=0.85)
ax.set_title('Distribucion Riesgo Compuesto AVR (Ley 1523/2012)', fontweight='bold')
ax.set_ylabel('N municipios'); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Dashboard Gestion del Riesgo — AVR + SPI + IVET',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

alto_riesgo = (df_avr['categoria_riesgo'].isin(['Alta','Muy Alta'])).sum()
print(f'Municipios en riesgo Alto/Muy Alto: {alto_riesgo}/{n_municipios}')
print(df_avr[df_avr['categoria_riesgo'].isin(['Alta','Muy Alta'])][['municipio','amenaza','vulnerabilidad','categoria_riesgo']])

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `gestion_riesgo` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para Gestión de Riesgo) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["precipitacion"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5c. LSTM para prediccion de caudales extremos — arquitectura y flujo

Las **Redes Neuronales Recurrentes (LSTM — Long Short-Term Memory)** son el estado del arte para prediccion de series temporales hidrologicas porque capturan dependencias temporales largas (memoria de lluvia acumulada, saturacion del suelo).

```
Arquitectura tipica para GRD:
  Input: [precipitacion_t-k, ..., t-1, SPI, pendiente, cobertura]
  LSTM Layer 1: 64 unidades, dropout=0.2
  LSTM Layer 2: 32 unidades, dropout=0.2
  Dense: 1 salida (caudal_t o prob_deslizamiento)
  Loss: MSE (caudal) o BCE (clasificacion amenaza)
```

Librerias: `keras` / `tensorflow` o `pytorch`. Referencia IDEAM: modelos LSTM calibrados con datos DHIME para cuencas andinas (RMSE ~15% del caudal medio).

> **InSAR / DInSAR (Interferometric SAR):** complementa el LSTM midiendo deformacion del terreno en mm/ano. Sentinel-1 permite generar mapas de velocidad de subsidencia o desplazamiento. Critico para monitorear deslizamientos activos antes del colapso.

In [ ]:
# Simulacion LSTM-like: ventana deslizante para prediccion de caudal maximo
# (numpy puro — sin dependencias de tensorflow/pytorch)
WINDOW = 6   # meses de contexto
HORIZON = 1  # meses a predecir

ts_vals = df_clean['precipitacion'].values if 'precipitacion' in df_clean.columns else df_clean.iloc[:, 0].values

# Construir matrices X, y con ventana deslizante (como LSTM)
X_lstm, y_lstm = [], []
for t in range(WINDOW, len(ts_vals) - HORIZON):
    X_lstm.append(ts_vals[t-WINDOW:t])  # contexto
    y_lstm.append(ts_vals[t:t+HORIZON]) # objetivo
X_lstm = np.array(X_lstm)  # shape: (muestras, WINDOW)
y_lstm = np.array(y_lstm).ravel()

# Regresion lineal como proxy del LSTM (misma ventana deslizante)
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
ridge = Ridge(alpha=1.0)
cv_rmse = (-cross_val_score(ridge, X_lstm, y_lstm,
                            cv=5, scoring='neg_root_mean_squared_error')).mean()
ridge.fit(X_lstm, y_lstm)
y_pred = ridge.predict(X_lstm)

# Plot: prediccion vs real
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(y_lstm, lw=1.5, color='#2980b9', label='Real')
ax1.plot(y_pred, lw=1.5, ls='--', color='#e74c3c', label='LSTM-proxy (Ridge, ventana=6m)')
ax1.set_title(f'LSTM-proxy — Prediccion caudal | CV-RMSE={cv_rmse:.2f}', fontweight='bold')
ax1.set_xlabel('Paso temporal'); ax1.set_ylabel('Precipitacion (mm)')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# SPI extremos
ax2.hist(df['spi'], bins=30, color='#3498db', alpha=0.7, edgecolor='white')
ax2.axvline(-1.5, color='red', ls='--', lw=2, label='Sequia severa')
ax2.axvline(1.5, color='green', ls='--', lw=2, label='Lluvia extrema')
ax2.set_title('Distribucion SPI — Deteccion de extremos hidrologicos', fontweight='bold')
ax2.set_xlabel('SPI'); ax2.set_ylabel('Frecuencia')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Ventana LSTM: {WINDOW} meses | CV-RMSE: {cv_rmse:.2f}')
print('Nota: LSTM real (TF/PyTorch) captura dependencias no lineales y memoria larga')
print('InSAR/DInSAR (Sentinel-1): monitoreo deformacion del terreno mm/ano complementa GRD')

## 5b. Análisis de excedencias normativas
> Compara `precipitacion` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["precipitacion"], variable="precipitacion")
if rep.empty:
    print("Sin normas colombianas registradas para 'precipitacion'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["precipitacion"], method="linear")
print(f"Faltantes antes: {df["precipitacion"].isna().sum()} | "
      f"después: {df_clean["precipitacion"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["precipitacion"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: gestion_riesgo -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Gestión del riesgo

- **Eventos extremos:** distribuciones de cola pesada (GEV / GPD), no normales — Mann-Kendall sobre máximos anuales.
- **Lag ENSO = 1 mes** (respuesta rápida en inundaciones / deslizamientos).
- **Umbrales determinísticos:** `AMENAZA_PRECIPITACION` (Ley 1523/2012, Decreto 1807/2014).
- **Período de retorno** es la métrica de gestión (BP-7) — convertir % excedencia a años.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Gestión de Riesgo (`gestion_riesgo`)
- **Variable analizada:** `precipitacion` (mm)
- **Modelos ejecutados:** SARIMA, SARIMAX, XGBOOST, RANDOM_FOREST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/gestion_riesgo.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.